# Python Subinterpreters: Una Nueva Era de Paralelismo

## Agenda

1. **Introducción y Contexto**
2. **El Problema del Paralelismo en Python**
3. **¿Qué son los Subinterpreters?**
4. **Evolución Histórica y PEPs**
5. **Arquitectura y Funcionamiento**
6. **Comparaciones de Performance**
7. **API y Uso Práctico**
8. **Casos de Uso Reales**
9. **Limitaciones y Desafíos**
10. **Futuro y Roadmap**

## El Desafío del Paralelismo en Python

### Problema Central
- Python es **inherentemente secuencial** debido al GIL
- Las máquinas modernas tienen **múltiples cores**
- Necesidad de **paralelismo real** para tareas CPU-intensivas
- Opciones actuales tienen **limitaciones significativas**

### Estado Actual

In [ ]:
# Threading: I/O bound solamente
# Multiprocessing: Alto overhead, comunicación compleja
# AsyncIO: Concurrencia, no paralelismo


## ¿Qué son los Subinterpreters?

### Definición
- **Subinterpretador**: Una copia completa del intérprete CPython
- Ejecuta **independientemente** del intérprete principal
- Cada uno tiene su **propio estado global**:
  - Tabla de nombres del scope global
  - Módulos importados
  - Sistema de garbage collection
  - **Su propio GIL** (desde Python 3.12)

## Subinterpreters vs Threading

In [ ]:
# Threading - Estado compartido
import threading
global_var = 0

def worker():
    global global_var
    global_var += 1  # ⚠️ Race condition posible

# Subinterpreters - Estado aislado  
interp1 = interpreters.create()
interp1.exec("global_var = 0")  # ✅ Independiente

interp2 = interpreters.create() 
interp2.exec("global_var = 100")  # ✅ No afecta interp1


## Historia y Evolución

### Timeline
- **Python 1.5 (1998)**: Subinterpreters via C-API únicamente
- **Python 3.12 (2023)**: PEP 684 - GIL por intérprete
- **Python 3.13 (2024)**: PEP 734/554 - API nativa de Python
- **Futuro**: APIs de alto nivel, mejor ecosystem support

### Estado Actual
- ✅ Funcional pero **experimental**
- ⚠️ API en evolución
- 🚧 Ecosystem support limitado

## PEP 684: A Per-Interpreter GIL

### Cambios Fundamentales

In [ ]:
// Antes: GIL global único
static PyThread_type_lock interpreter_lock;

// Después: GIL por intérprete
typedef struct _gil_runtime_state {
    PyThread_type_lock lock;
    // ... más estado por intérprete
} _gil_runtime_state;


### Beneficios
- **Paralelismo real** entre subinterpreters
- **Aislamiento** de estado garantizado
- **Compatibilidad** con código existente

## PEP 734/554: API de Python

### Objetivos
- Hacer subinterpreters **accesibles desde Python**
- Proporcionar **primitivas de comunicación**
- Base para **APIs de alto nivel** futuras

### Componentes Clave
- `interpreters` module
- `Queue` objects para comunicación
- `memoryview` para datos compartidos
- Sistema de tipos "shareables"

## Arquitectura Interna

### Estructura del Proceso

In [ ]:
Main Process
├── Main Interpreter (GIL 1)
│   ├── Thread 1
│   ├── Thread 2
│   └── Global State 1
├── Subinterpreter 1 (GIL 2)  
│   ├── Thread 3
│   ├── Thread 4
│   └── Global State 2
└── Subinterpreter 2 (GIL 3)
    ├── Thread 5
    └── Global State 3


## Comparación: Threading vs Multiprocessing vs Subinterpreters

| Aspecto | Threading | Multiprocessing | Subinterpreters |
|---------|-----------|-----------------|-----------------|
| **Paralelismo CPU** | ❌ (GIL) | ✅ | ✅ |
| **Overhead startup** | Muy Bajo | Muy Alto | Medio |
| **Comunicación** | Instantánea | Serialización | Queue nativo |
| **Memoria** | Compartida | Separada | Selectiva |
| **Aislamiento** | No | Completo | Controlado |
| **Debugging** | Complejo | Muy Complejo | Medio |

## Benchmarks: Tiempo de Startup

### Metodología

In [ ]:
# Lanzar 1000 workers en paralelo
def benchmark_startup(worker_type):
    start_time = time.perf_counter()
    # Crear workers...
    return time.perf_counter() - start_time


### Resultados

In [ ]:
Threading:           0.05s  ⚡ (pero sin paralelismo)
Subinterpreters:     1.10s  ✅ (5x más rápido que spawn)
Fork processes:      5.66s  ⚠️ (solo Linux, copy-on-write)
Spawn processes:   330.00s  ❌ (muy lento, pero universal)


## Benchmarks: Comunicación de Datos

### Test: 10 millones de enteros

In [ ]:
# Enviar datos entre workers
for i in range(10_000_000):
    queue.put(i)


### Resultados

In [ ]:
Baseline (single-thread):     0.8s  (referencia)
Subinterpreter Queue:         6.1s  ✅ (7x más rápido) 
Multiprocessing Queue:       43.0s  ❌ (serialización)

Overhead por item:          ~0.5ms  (aceptable)


## Tipos de Datos Compartibles

### Shareable Types

In [ ]:
# Definición de tipos compartibles
type Shareable = (
    str | bytes | int | float | bool | None |
    tuple[Shareable, ...] |  # Recursivo
    Queue |                  # Comunicación
    memoryview              # Arrays
)


### Ejemplos Prácticos

In [ ]:
# ✅ Permitidos
config = {"workers": 4, "timeout": 30}
data_view = memoryview(large_numpy_array)
comm_queue = interpreters.create_queue()

# ❌ No permitidos
custom_object = MyClass()
function_ref = my_function
class_ref = MyClass


## API Básica: Crear y Ejecutar

In [ ]:
from test.support import interpreters

# Crear subinterpretador
interp = interpreters.create()

# Verificar estado
print(f"Interpreter ID: {interp.id}")
print(f"Is running: {interp.is_running()}")

# Ejecutar código
interp.exec("""
import math
result = math.factorial(10)
print(f"10! = {result}")
""")

# Cerrar cuando termine
interp.close()


## API Avanzada: Preparación y Estado

In [ ]:
# Configurar estado inicial
shared_queue = interpreters.create_queue()
config_data = {"max_items": 1000, "timeout": 30}

interp = interpreters.create()
interp.prepare_main({
    "queue": shared_queue,
    "config": config_data,
    "worker_id": 42
})

# El código puede acceder a estas variables
interp.exec("""
print(f"Worker {worker_id} starting...")
print(f"Config: {config}")
# queue está disponible directamente
""")


## Comunicación: Queues

In [ ]:
from test.support.interpreters import queues

# Crear queues para comunicación bidireccional
task_queue = queues.create()
result_queue = queues.create()

# Worker code
worker_code = f"""
from test.support.interpreters import queues

tasks = queues.Queue({task_queue.id})
results = queues.Queue({result_queue.id})

while True:
    task = tasks.get()
    if task is None:  # Señal de terminación
        break
    
    # Procesar tarea
    result = task * task
    results.put(result)
"""

# Ejecutar worker en thread separado
interp = interpreters.create()
thread = threading.Thread(
    target=lambda: interp.exec(worker_code)
)
thread.start()


## Comunicación: Memoria Compartida

In [ ]:
import array
import ctypes

# Crear array grande
big_array = array.array("i", range(16_000_000))
pointer, size = big_array.buffer_info()

# Código para subinterpretador
worker_code = f"""
import ctypes

# Reconstruir array desde pointer
shared_array = (ctypes.c_int32 * {size}).from_address({pointer})

# Procesar chunk asignado
for i in range(start_idx, end_idx):
    shared_array[i] = shared_array[i] * 2
"""

# Distribuir trabajo
chunk_size = size // num_workers
for i in range(num_workers):
    interp = interpreters.create()
    interp.prepare_main({
        "start_idx": i * chunk_size,
        "end_idx": (i + 1) * chunk_size
    })
    interp.call_in_thread(worker_code)


## Patrón: Pool de Workers

In [ ]:
class SubinterpreterPool:
    def __init__(self, module: str, function: str, workers: int):
        self.tasks = interpreters.create_queue()
        self.results = interpreters.create_queue()
        self.workers = workers
        
        # Template code para workers
        self.worker_code = f"""
from {module} import {function}
from test.support.interpreters import queues

tasks = queues.Queue({self.tasks.id})
results = queues.Queue({self.results.id})

while True:
    task = tasks.get()
    if task is None:
        break
    
    try:
        result = {function}(task)
        results.put(('success', result))
    except Exception as e:
        results.put(('error', str(e)))
"""


## Pool de Workers: Implementación Completa

In [ ]:
class SubinterpreterPool:
    def worker(self):
        """Worker function que ejecuta en cada thread"""
        interp = interpreters.create()
        if self.shared_data:
            interp.prepare_main(**self.shared_data)
        interp.exec(self.worker_code)
        interp.close()
    
    def map(self, tasks):
        """Ejecutar tareas en paralelo"""
        # Iniciar workers
        threads = [
            threading.Thread(target=self.worker) 
            for _ in range(self.workers)
        ]
        for t in threads:
            t.start()
        
        # Enviar tareas
        for task in tasks:
            self.tasks.put(task)
        
        # Recolectar resultados
        results = []
        for _ in tasks:
            success, result = self.results.get()
            if success:
                results.append(result)
            else:
                raise Exception(result)
        
        # Terminar workers
        for _ in range(self.workers):
            self.tasks.put(None)
        
        for t in threads:
            t.join()
        
        return results


## Caso Real: Advent of Code

### Problema
- **Pathfinding algorithm** CPU-intensivo
- **11,000+ candidatos** para procesar
- Cada candidato requiere **simulación completa**

### Implementación

In [ ]:
def solve_candidate(start_pos, obstacle_pos, grid):
    """CPU-intensive pathfinding simulation"""
    # Complex algorithm...
    return has_loop

# Uso con subinterpreters
pool = SubinterpreterPool(
    module="pathfinding", 
    function="solve_candidate",
    workers=11,
    shared_data={"grid": grid_data}
)

candidates = [(start, obs) for obs in obstacle_positions]
results = pool.map(candidates)


## Resultados de Performance: Advent of Code

In [ ]:
# Tiempos de ejecución (11 cores disponibles)
workers=1:   3.58s  (baseline)
workers=2:   1.86s  (1.9x speedup)
workers=3:   1.28s  (2.8x speedup)
workers=4:   0.98s  (3.7x speedup)
workers=6:   0.74s  (4.8x speedup)
workers=8:   0.68s  (5.3x speedup)
workers=11:  0.60s  (6.0x speedup) ✅ Cerca del óptimo
workers=15:  0.60s  (sin mejora - saturación)


### Análisis
- **Speedup predecible** y escalable
- **Eficiencia alta** hasta el límite de cores
- **Sin overhead significativo** de comunicación

## Comparación: Free-Threading vs Subinterpreters

| Métrica | Free-Threading | Subinterpreters |
|---------|----------------|-----------------|
| **Estabilidad** | ⚠️ Muy experimental | ✅ Funcional |
| **Performance** | 🔄 Variable | ✅ Predecible |
| **Ecosystem** | ❌ Mínimo | ⚠️ Limitado |
| **Complejidad uso** | Alta | Media |
| **Overhead** | Bajo | Medio |
| **Debugging** | Muy difícil | Manejable |
| **Futuro** | Incierto | Prometedor |

### Recomendación Actual
**Subinterpreters** son más maduros y prácticos para uso real

## Limitaciones: Ecosystem Support

### Librerías Problemáticas

In [ ]:
# ❌ NumPy - Causa segfaults
import numpy as np  # Crash en subinterpreter

# ❌ Cython modules - Bloquean explícitamente  
import pandas as pd  # ImportError

# ❌ PyO3 (Rust) - No compatible
import pydantic  # No funciona

# ⚠️ ctypes - Soporte limitado
import ctypes  # Funciona parcialmente


### Módulos que SÍ Funcionan

In [ ]:
# ✅ Standard library
import math, json, re, time, os

# ✅ Pure Python packages
import requests, click

# ✅ Algunos C extensions actualizados
import array  # Funciona desde 3.13rc2


## Limitaciones: API y Usabilidad

### Restricciones Actuales

In [ ]:
# ❌ Solo strings, no funciones
def my_function(x):
    return x * 2

interp.exec(my_function)  # No funciona

# ✅ Workaround: importar en el subinterpreter
interp.exec("""
from mymodule import my_function
result = my_function(42)
""")


### Comunicación Limitada
- Solo tipos "shareable"
- Serialización necesaria para objetos complejos
- No hay shared objects nativos (todavía)

## Debugging y Observabilidad

### Desafíos

In [ ]:
# Debugging es más complejo
import pdb

def debug_worker():
    pdb.set_trace()  # ⚠️ Problemático en subinterpreter

# Logs pueden mezclarse
interp.exec("""
print("Message from subinterpreter")  # Sin contexto claro
""")


### Mejores Prácticas

In [ ]:
# Usar logging estructurado
worker_code = f"""
import logging
logger = logging.getLogger('worker_{worker_id}')
logger.info("Processing task", extra={{"worker_id": {worker_id}}})
"""

# Manejo explícito de errores
try:
    success, result = result_queue.get(timeout=30)
    if not success:
        logger.error(f"Worker error: {result}")
except queue.Empty:
    logger.error("Worker timeout")


## Performance: Overhead Analysis

### Startup Overhead

In [ ]:
# Costo por subinterpreter: ~1.1ms
# vs Thread: ~0.05ms  
# vs Process: 5.6ms (fork) / 330ms (spawn)

# Recomendación: Pool pattern
# ✅ Crear pocos subinterpreters, usar por mucho tiempo
# ❌ Crear/destruir frecuentemente


### Communication Overhead

In [ ]:
# Costo por mensaje: ~0.5ms
# Aceptable para:
# ✅ Tareas de >10ms
# ⚠️ Tareas de 1-10ms  
# ❌ Tareas de <1ms (usar threading)


## Casos de Uso Ideales

### ✅ Excelentes para Subinterpreters

In [ ]:
# 1. CPU-bound con poca comunicación
def process_images(image_list):
    return [heavy_image_processing(img) for img in image_list]

# 2. Simulaciones independientes
def monte_carlo_simulation(params):
    return run_simulation(params, iterations=1_000_000)

# 3. Procesamiento de arrays grandes
def process_array_chunk(array_view, start, end):
    for i in range(start, end):
        array_view[i] = complex_computation(array_view[i])


### ⚠️ Casos Problemáticos

In [ ]:
# Mucha comunicación inter-worker
# Dependencias NumPy/SciPy
# Tareas muy cortas (<10ms)
# Debugging intensivo necesario


## Estrategias de Migración

### Desde Threading

In [ ]:
# Antes: Threading (sin paralelismo real)
def threaded_approach():
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(cpu_task, data) 
                  for data in chunks]
        return [f.result() for f in futures]

# Después: Subinterpreters  
def subinterpreter_approach():
    pool = SubinterpreterPool("mymodule", "cpu_task", workers=4)
    return pool.map(chunks)


### Desde Multiprocessing

In [ ]:
# Evaluar trade-offs:
# ✅ Migrar si: startup overhead es problema
# ✅ Migrar si: comunicación frecuente
# ⚠️ Mantener si: ecosystem dependencies
# ⚠️ Mantener si: ya funciona bien


## Roadmap: Python 3.13+

### Mejoras Confirmadas
- **API más pythónica** para creación
- **Mejor error handling** 
- **Performance improvements**
- **Más módulos stdlib compatibles**

### En Desarrollo

In [ ]:
# Posible API futura más friendly
import concurrent.subinterpreters as subint

async def future_api():
    async with subint.Pool(workers=4) as pool:
        results = await pool.map(my_function, data)
    return results


## Roadmap: Largo Plazo

### Características Planeadas
- **Native shared objects** (no solo memoryview)
- **Better ecosystem integration**
- **Integration con asyncio**
- **Performance optimizations**

### Ejemplo Visión Futura

In [ ]:
# Hipotético: shared objects nativos
shared_dict = interpreters.SharedDict()
shared_list = interpreters.SharedList()

# Hipotético: asyncio integration  
async def async_subinterpreter():
    interp = await interpreters.create_async()
    result = await interp.exec_async(code)
    return result


## Comparación Final: Cuándo Usar Qué

### Decision Tree

In [ ]:
# ¿Es I/O bound?
if io_bound:
    use_asyncio()  # o threading

# ¿Es CPU bound?
elif cpu_bound:
    if needs_ecosystem_libraries:
        use_multiprocessing()
    elif high_communication_frequency:
        use_multiprocessing()  
    elif startup_performance_critical:
        use_subinterpreters()
    elif shared_array_processing:
        use_subinterpreters()
    else:
        use_multiprocessing()  # Más maduro

# ¿Es mixed workload?
else:
    consider_hybrid_approach()


## Mejores Prácticas

### Design Patterns

In [ ]:
# 1. Pool Pattern - Reutilizar subinterpreters
pool = SubinterpreterPool(workers=cpu_count())

# 2. Pipeline Pattern - Stages independientes  
stage1_pool = SubinterpreterPool("module1", "process", 4)
stage2_pool = SubinterpreterPool("module2", "transform", 4)

# 3. Shared Memory Pattern - Arrays grandes
shared_view = memoryview(large_array)
# Distribuir chunks a workers

# 4. Error Isolation - Fault tolerance
try:
    result = worker_pool.process(data)
except SubinterpreterError:
    # Reiniciar pool, logging, etc.
    handle_worker_failure()


## Conclusiones

### Estado Actual (2024)
- ✅ **Funcionalmente completo** para casos específicos
- ⚠️ **Ecosystem support limitado** pero mejorando
- ✅ **Performance predecible** y escalable
- ⚠️ **API en evolución** hacia más usabilidad

### Adopción Recomendada
- **Early adopters**: Experimentar en proyectos no críticos
- **Production**: Evaluar trade-offs vs multiprocessing
- **Largo plazo**: Prepararse para APIs mejoradas

### El Futuro del Paralelismo en Python
**Subinterpreters + Free-threading = Toolkit completo**
- Subinterpreters: Paralelismo robusto, casos específicos
- Free-threading: Paralelismo general, experimental
- Juntos: Cobertura completa de casos de uso

## Recursos y Referencias

### Documentación Oficial
- **PEP 684**: Per-Interpreter GIL
- **PEP 734**: Multiple Interpreters in Stdlib  
- **PEP 554**: Multiple Interpreters (legacy)

### Proyectos y Ejemplos
- **extrainterpreters**: Higher-level APIs
- **interpreters-backport**: Python 3.13 features en 3.12

### Benchmarks y Casos Reales
- Anthony Shaw's blog posts
- Jamie Chang's AOC implementation
- EuroPython 2024 talks

### Community
- python-dev mailing list
- GitHub discussions en CPython repo
- PyData conferences

## ¡Gracias!

### Preguntas y Discusión

**¿Listos para experimentar con el futuro del paralelismo en Python?**

In [ ]:
import concurrent.futures
import subinterpreters  # Coming soon!

# El futuro del paralelismo en Python
# está aquí... ¡Experimentemos juntos!


**Contact**: [Tu información de contacto]
**Slides**: [Enlace al repositorio]